# RSNA Knee — 05: Submission (runs on Kaggle's hidden test set, internet OFF)

**Settings:** Accelerator **GPU T4 x2** · Internet **Off** (required for submission) ·
Inputs: competition data + the output of notebook **04** (or the dataset holding your `model_f*.pt` files).

It preprocesses each test study exactly like notebook 03, runs every `model_f*.pt` it finds, averages them,
and writes `submission.csv`. When you commit it, it runs on the 3 sample test studies; when you **Submit**,
Kaggle reruns it on the ~1,300 hidden studies.

In [ ]:
# ---- shared preprocessing: identical code is used for training volumes and at submission time ----
import os, glob
import numpy as np
import pydicom
import cv2
from concurrent.futures import ThreadPoolExecutor

# 5 slots = plane x (fluid-sensitive / not). Axial non-fluid exists in only ~19% of studies -> used only as axial fallback.
SLOT_NAMES = ["sag_fs", "sag_nfs", "cor_fs", "cor_nfs", "ax_fs"]
SLOT_INDEX = {("sagittal", 1): 0, ("sagittal", 0): 1, ("coronal", 1): 2, ("coronal", 0): 3, ("axial", 1): 4}
N_SLICES = 20          # slices kept per slot (evenly spaced through the series)
IMG = 256              # output pixels per side
CROP_MM = 140.0        # constant physical field of view -> 0.55 mm/px at 256
MAX_SERIES_SLICES = 80 # prefer 2D series over very long 3D series when both exist

# Standard radiological display per plane, in DICOM patient coords (x = patient left, y = posterior, z = head):
# (axis that image columns run along, its sign, axis that image rows run along, its sign, through-plane slice axis)
TARGET = {"sagittal": (1, +1, 2, -1, 0),   # front of knee on image left, head up
          "coronal":  (0, +1, 2, -1, 1),   # patient's right on image left, head up
          "axial":    (0, +1, 1, +1, 2)}   # patient's right on image left, front up

def plane_of(iop):
    v = np.asarray(iop, dtype=float)
    n = np.cross(v[:3], v[3:6])
    return ["sagittal", "coronal", "axial"][int(np.argmax(np.abs(n)))]

def _read_header(f):
    try:
        ds = pydicom.dcmread(f, stop_before_pixels=True,
                             specific_tags=["ImageOrientationPatient", "ImagePositionPatient", "PixelSpacing",
                                            "Rows", "Columns", "InstanceNumber"])
        return f, [float(x) for x in ds.ImageOrientationPatient], [float(x) for x in ds.ImagePositionPatient], \
               [float(x) for x in ds.PixelSpacing]
    except Exception:
        return f, None, None, None

def _read_pixels(f):
    try:
        ds = pydicom.dcmread(f)
        a = ds.pixel_array.astype(np.float32)
        if a.ndim == 3:  # multi-frame or colour: take first frame / channel
            a = a[0] if a.shape[0] < a.shape[-1] else a[..., 0]
        return a, str(ds.file_meta.TransferSyntaxUID) if hasattr(ds, "file_meta") else "?"
    except Exception as ex:
        return None, type(ex).__name__

def series_headers(series_dir, pool):
    files = sorted(glob.glob(os.path.join(series_dir, "*.dcm"))) or \
            sorted(f for f in glob.glob(os.path.join(series_dir, "*")) if os.path.isfile(f))
    hdrs = [h for h in pool.map(_read_header, files) if h[1] is not None]
    if not hdrs:
        return None
    # keep the majority orientation (drops localisers mixed into a series)
    keys = [tuple(np.round(h[1], 1)) for h in hdrs]
    best = max(set(keys), key=keys.count)
    hdrs = [h for h, k in zip(hdrs, keys) if k == best]
    plane = plane_of(hdrs[0][1])
    hdrs.sort(key=lambda h: h[2][TARGET[plane][4]])
    return {"plane": plane, "hdrs": hdrs, "n": len(hdrs)}

def reorient(img, r, c, spacing, plane):
    """Flip/transpose so every series is displayed in the same standard orientation."""
    ra, rs, ca, cs, _ = TARGET[plane]
    r, c = np.asarray(r), np.asarray(c)
    sp = list(spacing)  # [row spacing (between rows), column spacing]
    if int(np.argmax(np.abs(r))) != ra:
        img, r, c, sp = img.T, c, r, sp[::-1]
    if np.sign(r[ra]) != rs:
        img = img[:, ::-1]
    if np.sign(c[ca]) != cs:
        img = img[::-1, :]
    return np.ascontiguousarray(img), sp

def crop_resize(img, sp):
    h, w = img.shape
    hr, hc = CROP_MM / 2 / max(sp[0], 1e-3), CROP_MM / 2 / max(sp[1], 1e-3)
    cy, cx = h / 2, w / 2
    y0, y1, x0, x1 = int(round(cy - hr)), int(round(cy + hr)), int(round(cx - hc)), int(round(cx + hc))
    pad = max(0, -y0, -x0, y1 - h, x1 - w)
    if pad:
        img = np.pad(img, pad)
        y0, y1, x0, x1 = y0 + pad, y1 + pad, x0 + pad, x1 + pad
    crop = img[y0:y1, x0:x1]
    interp = cv2.INTER_AREA if crop.shape[0] > IMG else cv2.INTER_LINEAR
    return cv2.resize(crop, (IMG, IMG), interpolation=interp)

def build_slot(info, pool):
    hdrs = info["hdrs"]
    idx = np.round(np.linspace(0, len(hdrs) - 1, N_SLICES)).astype(int)
    uniq = sorted(set(idx.tolist()))
    pix = dict(zip(uniq, pool.map(lambda i: _read_pixels(hdrs[i][0]), uniq)))
    out, ts = [], set()
    for i in idx:
        a, t = pix[i]
        ts.add(t)
        if a is None:
            out.append(np.zeros((IMG, IMG), np.float32)); continue
        _, iop, _, sp = hdrs[i]
        a, sp2 = reorient(a, iop[:3], iop[3:], sp, info["plane"])
        out.append(crop_resize(a, sp2))
    vol = np.stack(out)
    nz = vol[vol > 0]
    lo, hi = (np.percentile(nz, [1, 99.5]) if nz.size > 100 else (0.0, 1.0))
    vol = np.clip((vol - lo) / max(hi - lo, 1e-6), 0, 1) * 255
    return vol.astype(np.uint8), ts, sum(pix[i][0] is None for i in uniq)

def process_study(study_dir, series_rows):
    """series_rows: list of (SeriesInstanceUID, Fluid_Sensitive). Returns (vol [5,N,IMG,IMG] uint8, mask [5], info)."""
    vol = np.zeros((len(SLOT_NAMES), N_SLICES, IMG, IMG), np.uint8)
    mask = np.zeros(len(SLOT_NAMES), np.uint8)
    info = {"n_series": len(series_rows), "plane_mismatch": 0, "bad_pixels": 0, "ts": set(), "slot_n": [0] * 5}
    cands = {}
    with ThreadPoolExecutor(8) as pool:
        for uid, fluid in series_rows:
            sdir = os.path.join(study_dir, str(uid))
            if not os.path.isdir(sdir):
                hits = [d for d, _, _ in os.walk(study_dir) if os.path.basename(d) == str(uid)]
                if not hits:
                    continue
                sdir = hits[0]
            h = series_headers(sdir, pool)
            if h is None:
                continue
            key = (h["plane"], int(fluid))
            slot = SLOT_INDEX.get(key, 4 if h["plane"] == "axial" else None)
            if slot is None:
                continue
            fallback = key not in SLOT_INDEX
            score = (not fallback, h["n"] <= MAX_SERIES_SLICES, -abs(h["n"] - 30))
            if slot not in cands or score > cands[slot][0]:
                cands[slot] = (score, h)
        for slot, (_, h) in cands.items():
            v, ts, bad = build_slot(h, pool)
            vol[slot], mask[slot] = v, 1
            info["ts"] |= ts
            info["bad_pixels"] += bad
            info["slot_n"][slot] = h["n"]
    return vol, mask, info

In [ ]:
# ---- shared model code: identical in training and submission notebooks ----
import math
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import timm
import torch.utils.checkpoint

LABELS = ["ACL", "MCL", "Medial Meniscus", "Lateral Meniscus", "Medial OA", "Lateral OA", "PF OA",
          "Effusion", "Synovitis", "Baker's", "Contusion", "Fracture"]

def to_25d(x, slice_idx):
    """x: [B, S, N, H, W] uint8 -> [B, S, n, 3, H, W] float, channels = slices (i-1, i, i+1)."""
    N = x.shape[2]
    idx = torch.as_tensor(slice_idx, device=x.device)
    trip = torch.stack([(idx - 1).clamp(0, N - 1), idx, (idx + 1).clamp(0, N - 1)], 1)  # [n, 3]
    out = x[:, :, trip]  # [B, S, n, 3, H, W]
    return out.float().div_(255.0).sub_(0.5).div_(0.25)

class KneeNet(nn.Module):
    def __init__(self, backbone="efficientnet_b0.ra_in1k", pretrained=True, n_slots=5, n_slices=20,
                 d=256, n_layers=2, n_labels=12, drop=0.2, grad_ckpt=True, freeze_blocks=0, img_size=None):
        super().__init__()
        kw = {"img_size": img_size} if img_size else {}
        self.enc = timm.create_model(backbone, pretrained=pretrained, num_classes=0, in_chans=3, **kw)
        self.is_vit = hasattr(self.enc, "blocks") and hasattr(self.enc, "patch_embed")
        self.img_size = img_size
        self.freeze_blocks = freeze_blocks if self.is_vit else 0
        self.grad_ckpt = grad_ckpt
        if self.is_vit:
            # DINOv2-style: frozen patch embedding + first k blocks, trainable rest; feature = [CLS, mean(patches)]
            for prm in list(self.enc.patch_embed.parameters()) + [self.enc.pos_embed] + \
                       ([self.enc.cls_token] if getattr(self.enc, "cls_token", None) is not None else []):
                prm.requires_grad_(self.freeze_blocks == 0)
            for blk in self.enc.blocks[:self.freeze_blocks]:
                for prm in blk.parameters():
                    prm.requires_grad_(False)
            nf = 2 * self.enc.num_features
        else:
            if grad_ckpt and hasattr(self.enc, "set_grad_checkpointing"):
                self.enc.set_grad_checkpointing(True)
            nf = self.enc.num_features
        self.proj = nn.Sequential(nn.Linear(nf, d), nn.LayerNorm(d))
        self.slot_emb = nn.Parameter(torch.zeros(n_slots, 1, d))
        self.pos_emb = nn.Parameter(torch.zeros(1, n_slices, d))
        nn.init.trunc_normal_(self.slot_emb, std=0.02); nn.init.trunc_normal_(self.pos_emb, std=0.02)
        layer = nn.TransformerEncoderLayer(d, nhead=8, dim_feedforward=2 * d, dropout=0.1, batch_first=True, norm_first=True)
        self.mixer = nn.TransformerEncoder(layer, n_layers)
        self.queries = nn.Parameter(torch.randn(n_labels, d) * 0.02)
        self.attn = nn.MultiheadAttention(d, 8, dropout=0.1, batch_first=True)
        self.drop = nn.Dropout(drop)
        self.head_w = nn.Parameter(torch.zeros(n_labels, d))
        self.head_b = nn.Parameter(torch.zeros(n_labels))

    def encode(self, imgs):
        if not self.is_vit:
            return self.enc(imgs)
        enc, k = self.enc, self.freeze_blocks
        if self.img_size and imgs.shape[-1] != self.img_size:   # centre-crop (e.g. 256 -> 252, a multiple of 14)
            o = (imgs.shape[-1] - self.img_size) // 2
            imgs = imgs[..., o:o + self.img_size, o:o + self.img_size]
        with torch.set_grad_enabled(torch.is_grad_enabled() and k == 0):
            t = enc.patch_embed(imgs)
            t = enc._pos_embed(t)
            t = enc.norm_pre(t)
            for blk in enc.blocks[:k]:
                t = blk(t)
        for blk in enc.blocks[k:]:
            if self.grad_ckpt and self.training and torch.is_grad_enabled():
                t = torch.utils.checkpoint.checkpoint(blk, t, use_reentrant=False)
            else:
                t = blk(t)
        t = enc.norm(t)
        npt = getattr(enc, "num_prefix_tokens", 1)
        return torch.cat([t[:, 0], t[:, npt:].mean(1)], 1)

    def forward(self, x, mask, slice_idx):
        """x: [B, S, N, H, W] uint8 on device; mask: [B, S] (1 = slot present); slice_idx: slices to use."""
        B, S = x.shape[:2]
        n = len(slice_idx)
        imgs = to_25d(x, slice_idx)                                  # [B, S, n, 3, H, W]
        feats = self.encode(imgs.flatten(0, 2))                      # [B*S*n, nf]
        t = self.proj(feats).view(B, S, n, -1)
        pos = self.pos_emb[:, torch.as_tensor(slice_idx, device=x.device)]  # [1, n, d]
        t = t + pos.unsqueeze(0) + self.slot_emb.unsqueeze(0)       # [B, S, n, d]
        t = t.flatten(1, 2)                                          # [B, S*n, d]
        pad = (mask == 0).repeat_interleave(n, dim=1)                # [B, S*n] True = ignore
        pad = pad & ~pad.all(1, keepdim=True)                        # a study with no slots: attend to zeros
        t = self.mixer(t, src_key_padding_mask=pad)
        q = self.queries.unsqueeze(0).expand(B, -1, -1)
        o, _ = self.attn(q, t, t, key_padding_mask=pad)              # [B, 12, d]
        o = self.drop(o)
        return (o * self.head_w).sum(-1) + self.head_b               # [B, 12] logits

SAG_SLOTS, INPLANE_X_SLOTS = [0, 1], [2, 3, 4]   # sagittal slices run along x; coronal/axial columns run along x

def mirror(x):
    """Mirror whole studies left<->right (patient x axis). x: [B, S, N, H, W].
    Anatomy is preserved (a mirrored right knee is a valid left knee), so labels do not change."""
    x = x.clone()
    x[:, SAG_SLOTS] = x[:, SAG_SLOTS].flip(2)          # reverse sagittal slice order
    x[:, INPLANE_X_SLOTS] = x[:, INPLANE_X_SLOTS].flip(4)  # flip image columns
    return x

def predict_tta(model, x, mask, slice_idx, tta=True):
    p = torch.sigmoid(model(x, mask, slice_idx).float())
    if tta:
        p = (p + torch.sigmoid(model(mirror(x), mask, slice_idx).float())) / 2
    return p

def slice_indices(n_total, n_use):
    return np.round(np.linspace(0, n_total - 1, n_use)).astype(int).tolist()

In [ ]:
import os, glob, time
import pandas as pd
from concurrent.futures import ProcessPoolExecutor

INPUT_ROOT = os.environ.get("KNEE_INPUT_ROOT", "/kaggle/input")
OUT = os.environ.get("KNEE_OUT_DIR", "/kaggle/working")
BATCH = 4
TTA = True   # average original + mirrored prediction
t0 = time.time()

def find(name):
    for cur, dirs, files in os.walk(INPUT_ROOT, followlinks=True):
        if len(dirs) > 200:
            dirs[:] = []
        if name in files:
            return os.path.join(cur, name)

DATA = os.path.dirname(find("test_series.csv"))
test_ser = pd.read_csv(os.path.join(DATA, "test_series.csv"))
sample = pd.read_csv(os.path.join(DATA, "sample_submission.csv"))
test_ids = pd.read_csv(os.path.join(DATA, "test.csv")).StudyInstanceUID.tolist()
groups = {s: list(zip(g.SeriesInstanceUID, g.Fluid_Sensitive)) for s, g in test_ser.groupby("StudyInstanceUID")}
IMG_ROOT = os.path.join(DATA, "test_series")
print(f"test studies: {len(test_ids)}")

dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")
ckpts = sorted(glob.glob(os.environ.get("KNEE_MODEL_GLOB", f"{INPUT_ROOT}/**/model_f*.pt"), recursive=True))
assert ckpts, "No model_f*.pt found — attach notebook 04's output (or your model dataset) as an input."
models, sidx_list = [], []
for c in ckpts:
    ck = torch.load(c, map_location="cpu", weights_only=False)
    cf = ck["cfg"]
    m = KneeNet(cf["backbone"], pretrained=False, grad_ckpt=False, freeze_blocks=cf.get("freeze_blocks", 0),
                img_size=cf.get("img_size") if ("vit" in cf["backbone"] or "dinov2" in cf["backbone"]) else None)
    m.load_state_dict(ck["state_dict"]); m.to(dev).eval()
    models.append(m); sidx_list.append(slice_indices(N_SLICES, ck["cfg"]["n_slices_use"]))
    print("loaded", os.path.relpath(c, INPUT_ROOT), "| backbone", ck["cfg"]["backbone"], "| epoch", ck.get("epoch"))
amp_dtype = torch.float16 if dev.type == "cuda" else torch.bfloat16

def prep(uid):
    try:
        vol, mask, _ = process_study(os.path.join(IMG_ROOT, uid), groups.get(uid, []))
    except Exception:
        vol = np.zeros((len(SLOT_NAMES), N_SLICES, IMG, IMG), np.uint8); mask = np.zeros(len(SLOT_NAMES), np.uint8)
    return uid, vol, mask

In [ ]:
preds = {}
def flush(batch):
    x = torch.from_numpy(np.stack([b[1] for b in batch])).to(dev)
    mk = torch.from_numpy(np.stack([b[2] for b in batch]).astype(np.float32)).to(dev)
    with torch.no_grad(), torch.autocast(dev.type, dtype=amp_dtype, enabled=dev.type == "cuda"):
        p = np.mean([predict_tta(m, x, mk, s, TTA).cpu().numpy() for m, s in zip(models, sidx_list)], 0)
    for (uid, _, msk), row in zip(batch, p):
        preds[uid] = row if msk.sum() > 0 else None

batch = []
with ProcessPoolExecutor(max_workers=os.cpu_count()) as ex:
    for i, item in enumerate(ex.map(prep, test_ids)):
        batch.append(item)
        if len(batch) == BATCH:
            flush(batch); batch = []
        if (i + 1) % 100 == 0:
            print(f"  {i+1}/{len(test_ids)}  {(time.time()-t0)/60:.1f} min")
if batch:
    flush(batch)

ok = [v for v in preds.values() if v is not None]
fallback = np.mean(ok, 0) if ok else np.full(len(LABELS), 0.3)
rows = [{"StudyInstanceUID": u, **dict(zip(LABELS, preds.get(u) if preds.get(u) is not None else fallback))} for u in test_ids]
sub = pd.DataFrame(rows)[sample.columns]
os.makedirs(OUT, exist_ok=True)
sub.to_csv(os.path.join(OUT, os.environ.get("KNEE_OUT_NAME", "submission.csv")), index=False)
print(sub.shape, "| studies with no usable images:", sum(v is None for v in preds.values()),
      f"| total {(time.time()-t0)/60:.1f} min")
print(sub.head())